# Step 2: 不该量化什么（工业经验法则）

**目标**：掌握工业界**几乎总是该回退**的层类清单——`lm_head`、embedding、MoE router、(部分)Norm、特定 `down_proj`——以及它们的「为什么」。s1 给的是**数据驱动**的敏感度（要 profile 才知道）；本节给的是**先验**：不 profile 也能先把这些层排除掉。两者结合 = 又快又稳的调优起点。

**对应 OUTLINE 课时**：3.2「不该量化什么」工业经验法则（~40 分钟）。


## 学完应能讲清（学完本节应能口头回答）

1. `lm_head` 为什么**几乎总是**要回退？给一个具体理由（它是分类头，量化误差直接放大到 logits）。
2. embedding 和 lm_head 在 Qwen2.5 里可能**共享权重**（tie），共享权重回退时要注意什么？（回退一个等于回退两个）
3. 为什么「首末层回退」**不是铁律**？OUTLINE 给的反例是什么？（Qwen2.5 实测首末层未必最敏感，得 profile）
4. Norm 层（RMSNorm/LayerNorm）参数极少，量化它「收益小风险大」——这个权衡的数学直觉是什么？（参数少→压缩省不了多少；量化引入的尺度误差却影响整层激活）
5. 把经验法则和 s1 的数据驱动 profiling 结合，正确的调优顺序是什么？（先套经验法则回退铁定项，再用 profiling 找剩余敏感层）


In [ ]:
%%capture
import pathlib, os, re
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()
from transformers import Qwen2Config, Qwen2ForCausalLM
from llmcompressor.modifiers.quantization import QuantizationModifier


In [ ]:
# Setup cell（双 env：模块根 = 含 scripts/ + steps/ 的 course/m3-tuning-eval/）。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT      = _find_module_root(pathlib.Path.cwd())
MODEL_DIR        = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR   = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT         = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT =", MODULE_ROOT, "| 0.5B @", TINY_MODEL_DIR.exists())


## 原理：哪些层「量化收益小、风险大」

OUTLINE 3.2 经验共识（工业界反复验证）：

| 层类 | 该不该量化 | 为什么 |
|---|---|---|
| `lm_head` | **几乎总是回退** | 分类头，量化误差直接进 logits/概率分布，下游任务（生成/分类）最敏感 |
| embedding (`embed_tokens`) | **不量化** | 词表嵌入，离群点常见；且常与 lm_head tie 共享 |
| MoE router / gate | **回退** | 量化后路由逻辑崩（选错专家） |
| RMSNorm / LayerNorm | **通常不量化** | 参数极少（一个 hidden_size 向量），量化省的显存可忽略，但尺度误差影响整层激活 |
| 首末几层 transformer block | **视情况** | OUTLINE 易错点：**非铁律**——Qwen2.5 实测首末层未必最敏感，旋转位置编码下 v_proj/k_proj 可能更敏感，**必须 profile 确认** |
| 某些模型 `down_proj` | **视情况** | 低比特（W4）下常见敏感，W8 通常还好 |

**关键认知**：经验法则是**先验**，给「不用想就回退」的清单；s1 的 profiling 是**后验**，给「测出来才回退」的清单。工程上**先用经验法则省事、再用 profiling 精修**。


### 端到端：经验法则在 layer fallback 全流程的位置

经验法则 = 「便宜的第一刀」：在不做任何 profiling 的前提下，先把铁定要回退的层（lm_head/embedding/Norm）加进 `ignore`，立刻拿到一个「不太差」的基线。再叠加 s1 的敏感度排序做精修。这是为什么 s2 在 s1 之后讲——s1 给精度，s2 给效率。


## 亲手摸一摸：经验法则清单 vs 模型实际层结构

建个 tiny Qwen2，把它的所有层名过一遍「经验法则」，看哪些会被自动判为「该回退」。


In [ ]:
## 摸一摸：列出 tiny 模型所有模块，手动对照经验法则
tiny = Qwen2ForCausalLM(Qwen2Config(
    num_hidden_layers=2, hidden_size=64, intermediate_size=128,
    num_attention_heads=4, num_key_value_heads=2, vocab_size=320, tie_word_embeddings=False))
print("=== 经验法则「铁定回退」清单 ===")
for name, mod in tiny.named_modules():
    short = name.split('.')[-1] or name
    is_linear = isinstance(mod, nn.Linear)
    tag = []
    if name == "lm_head": tag.append("lm_head(铁律回退)")
    if "embed_tokens" in name: tag.append("embedding(不量化)")
    if isinstance(mod, (nn.RMSNorm, nn.LayerNorm)): tag.append("Norm(通常不量化)")
    if is_linear and tag:
        print(f"  {name:45s} -> {' | '.join(tag)}")
print("\nlm_head 是 nn.Linear：", isinstance(tiny.lm_head, nn.Linear))
print("tie_word_embeddings：", tiny.config.tie_word_embeddings, "（False=lm_head 与 embed_tokens 独立）")


## 本步填空

1. **`should_skip_layer(name, module, tie_word_embeddings)`**（判断型）—— 给定层名+模块对象，按经验法则返回 `True`（该回退/不量化）或 `False`。**为什么这么设计（填前先想）**：先验判据要**确定性强**（lm_head/embedding/Norm 几乎 100% 该回退），但「首末层」这种**非铁律**的绝不能写死——得留给 s1 的 profile。
2. **`classify_layer_risk(name, module)`** —— 把层分到三档风险（`always_skip`/`profile_first`/`safe_to_quantize`），把经验法则和「需 profile」区分开。**为什么这么设计**：现实调优要把层分成「不用想就回退」「得测」「放心量化」三类，分别进 ignore / 进 profile 队列 / 进量化 targets。


In [ ]:
def should_skip_layer(name, module, tie_word_embeddings=False):
    """判断型：按工业经验法则，这个层是否该回退（不量化）。
    返回 True = 该回退。

    经验法则（OUTLINE 3.2 铁律项，确定性强）：
      - lm_head：分类头，几乎总是回退。
      - embed_tokens：词表嵌入，不量化（若 tie_word_embeddings=True，它和 lm_head 共享，
        回退一个等于回退两个——这里对 embedding 也直接返回 True）。
      - Norm 层（RMSNorm / LayerNorm）：参数极少，量化收益小风险大，不量化。
      - MoE router/gate（名字含 'gate' 且非 mlp.gate_proj 的 router）：回退。
    注意：「首末 transformer block」是**非铁律**——不在此函数判（留给 s1 profile）。
    """
    # TODO: 实现上述四条铁律判断，返回 bool。
    #   提示1：lm_head 用 name == "lm_head"（不是 "lm_head" in name，避免误伤）。
    #   提示2：Norm 用 isinstance(module, (nn.RMSNorm, nn.LayerNorm))。
    #   提示3：embedding 用 "embed_tokens" in name。
    #   提示4：MoE router 一般名字含 'gate' 但不是 mlp 的 gate_proj——课程模型非 MoE，
    #          这条可用 name.endswith("gate") 且 "mlp" not in name 近似（写上体现完整性）。
    raise NotImplementedError


In [ ]:
def classify_layer_risk(name, module):
    """把层分到三档风险之一：
      'always_skip'       —— 铁律回退（lm_head/embedding/Norm/MoE router）。
      'profile_first'     —— 非铁律但常见敏感，必须先 profile（首末 transformer block、down_proj）。
      'safe_to_quantize'  —— 经验上可放心量化的多数 attention/MLP 投影层。

    为什么这么设计（填前先想）：把「先验确定的回退」「需后验确认的」「默认安全」分开，
    调优时 always_skip 直接进 ignore、profile_first 进 s1 的敏感度扫描队列、safe_to_quantize 进量化 targets。
    """
    # TODO: 实现：
    #   1) 先调 should_skip_layer(name, module)，True -> 'always_skip'。
    #   2) 否则若层在首/末 transformer block（name 形如 model.layers.0.* 或 .{N-1}.*），
    #      或名字以 'down_proj' 结尾 -> 'profile_first'。
    #      （首末层号怎么拿：从 name 里 parse 出 layer id，需要总层数——本函数为判断型，
    #       可用启发式：name.split('.')[2] in {'0'} 判首层；末层留给调用方传 total_layers，
    #       这里对 down_proj 一律 profile_first 即可体现规则。）
    #   3) 否则（多数 q/k/v/o/gate/up_proj）-> 'safe_to_quantize'。
    #   非 Linear 的其它层（无意义）返回 'always_skip' 兜底（不量化非权重层）。
    raise NotImplementedError


In [ ]:
def _tiny():
    return Qwen2ForCausalLM(Qwen2Config(
        num_hidden_layers=2, hidden_size=64, intermediate_size=128,
        num_attention_heads=4, num_key_value_heads=2, vocab_size=320, tie_word_embeddings=False))

def test_should_skip_lm_head_and_embedding():
    m = _tiny()
    assert should_skip_layer("lm_head", m.lm_head) is True
    assert should_skip_layer("model.embed_tokens", m.model.embed_tokens) is True

def test_should_skip_norm():
    m = _tiny()
    for name, mod in m.named_modules():
        if isinstance(mod, (nn.RMSNorm, nn.LayerNorm)):
            assert should_skip_layer(name, mod) is True, f"Norm 应回退: {name}"

def test_should_not_skip_normal_linear():
    m = _tiny()
    q = m.get_submodule("model.layers.1.self_attn.q_proj")
    assert should_skip_layer("model.layers.1.self_attn.q_proj", q) is False

def test_classify_three_buckets():
    m = _tiny()
    assert classify_layer_risk("lm_head", m.lm_head) == "always_skip"
    assert classify_layer_risk("model.layers.0.mlp.down_proj",
                               m.get_submodule("model.layers.0.mlp.down_proj")) == "profile_first"
    assert classify_layer_risk("model.layers.1.self_attn.q_proj",
                               m.get_submodule("model.layers.1.self_attn.q_proj")) == "safe_to_quantize"
    # 首层（layer 0）的任意投影层 -> profile_first（首末层非铁律但需测）
    assert classify_layer_risk("model.layers.0.self_attn.q_proj",
                               m.get_submodule("model.layers.0.self_attn.q_proj")) == "profile_first"

# L1 必过守卫：ipytest.run 返回 pytest 退出码；非 0（有测试失败）→ 抛异常让 nbconvert 真挂。
# （用 ipytest.run() 而非 %%ipytest magic：magic 吞掉失败、exit_code 属性在本版不可靠。）
_ec = ipytest.run("-qq")
assert _ec == 0, f"L1 测试未全过（exit_code={_ec}），见上方 pytest 输出。"

## L2（CPU 概念）：经验法则套到 tiny 模型，生成「初版 ignore 清单」

把 `should_skip_layer` 套到 tiny 模型所有 Linear，生成一个**不用 profile 就能得到的 ignore 清单**——这就是「便宜的第一刀」。注意它不会把首末层写死回退（那要留给 s1）。


In [ ]:
## L2：从经验法则生成初版 ignore 清单
tiny = _tiny()
rule_based_ignore = sorted({
    name.split('.')[-1] if name.split('.')[-1] else name
    for name, mod in tiny.named_modules()
    if isinstance(mod, nn.Linear) and should_skip_layer(name, mod)
})
print("经验法则初版 ignore（按层类名，非逐层）：", rule_based_ignore)
assert "lm_head" in rule_based_ignore, "lm_head 必在经验 ignore 里"

# 三档分布统计
from collections import Counter
buckets = Counter(classify_layer_risk(n, m) for n, m in tiny.named_modules() if isinstance(m, nn.Linear))
print("\n三档风险分布：", dict(buckets))
print("  always_skip 直接进 ignore；profile_first 进 s1 扫描队列；safe_to_quantize 进量化 targets")
print("\nL2 通过：经验法则清单生成正确，lm_head 铁律回退。")


## L3（H200，GPU 守卫）：套经验法则到真 7B，看 ignore 清单规模

本节偏概念，L3 只是把 `should_skip_layer` 套到真 7B 的完整模块树（不需要重量化），看经验法则会回退多少层——感受「不用 profile 就能省下多少调试」。


In [ ]:
import torch, os
def run_l3_rule_check():
    model = Qwen2ForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.float16, device_map="auto")
    rule_skip = [n for n, m in model.named_modules() if isinstance(m, nn.Linear) and should_skip_layer(n, m)]
    from collections import Counter
    buckets = Counter(classify_layer_risk(n, m) for n, m in model.named_modules() if isinstance(m, nn.Linear))
    print(f"7B 经验法则回退 Linear 层数 = {len(rule_skip)}")
    print(f"三档分布 = {dict(buckets)}")
    __import__("json").dump({"rule_based_ignore": rule_skip, "buckets": dict(buckets)},
                            open(OUT_ROOT / "s2_rule_ignore.json", "w"), indent=2)
    print("已存 out/s2_rule_ignore.json")

if torch.cuda.is_available() and not os.environ.get("SKIP_L3"):
    run_l3_rule_check()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（本节偏概念，L2 已覆盖核心逻辑）")


## 产物检查


In [ ]:
import json
p = OUT_ROOT / "s2_rule_ignore.json"
if p.exists():
    d = json.loads(p.read_text())
    print("7B 经验 ignore 规模：", len(d["rule_based_ignore"]), "层")
    print("三档：", d["buckets"])
else:
    print(f"{p} 不存在（L3 未跑或被 SKIP_L3 跳过）")
